# Feature preprocessing
Run from `notebooks/` after rebuilding the DATA-05 datasets. These steps use the
[reviewed feature contract](../data/README.md#core-and-expanded-feature-sets).
The command `python prepare_features.py` runs the same steps. No model is trained
here, and the 2024 test files are not read.

In [ ]:
from pathlib import Path
import sys

if Path.cwd().name != "notebooks":
    raise RuntimeError("Run this notebook from notebooks/.")
sys.path.insert(0, str(Path.cwd().parent))
from src.data.preprocess import notebook_root
from src.features.prepare import load_feature_split, save_feature_artifacts
from src.features.preprocessing import make_preprocessor

PROJECT_ROOT = notebook_root()
frames, targets, exclusions = {}, {}, {}

## 1. Construct training features
Check source hashes, keys, and codes. Aggregate vehicles and drivers by crash,
then exclude unknown target rows. Missing driver features do not drop crashes.

In [ ]:
frames["train"], targets["train"], exclusions["train"] = load_feature_split(PROJECT_ROOT, "train")
print(exclusions["train"])
frames["train"].head()

## 2. Construct validation features
Apply the same fixed construction rules. This does not fit anything or use
validation outcomes to choose features.

In [ ]:
frames["validation"], targets["validation"], exclusions["validation"] = load_feature_split(PROJECT_ROOT, "validation")
print("Validation features:", frames["validation"].shape)

## 3. Fit preprocessing on training only
Categorical missing values use a separate -1 value before one-hot encoding.
Numeric values use training medians and scaling. All-empty numeric training
columns stay present with a zero fallback. Neither keys nor labels enter the
transformed matrices. Each configuration gets its own fitted pipeline.

In [ ]:
preprocessors, matrices = {}, {}
for configuration in ("core", "expanded"):
    preprocessors[configuration] = make_preprocessor(configuration)
    matrices[configuration] = {
        "train": preprocessors[configuration].fit_transform(frames["train"])
    }
    print(configuration, matrices[configuration]["train"].shape)

## 4. Transform validation
Reuse the training medians, scales, and category vocabulary. Unseen categories
produce an all-zero block for that field; columns stay in the same order.

In [ ]:
for configuration, preprocessor in preprocessors.items():
    matrices[configuration]["validation"] = preprocessor.transform(frames["validation"])
    assert matrices[configuration]["train"].shape[1] == matrices[configuration]["validation"].shape[1]
    print(configuration, matrices[configuration]["validation"].shape)

## 5. Save the reusable artifacts
Save matrices, matching crash keys and labels, encoded column names, fitted
pipelines, and metadata under `data/processed/features/`. These files are local
and ignored by git. The next baseline tasks consume them. The expanded behavior
features remain retrospective-only; these exports do not establish early prediction.

In [ ]:
for configuration, preprocessor in preprocessors.items():
    save_feature_artifacts(PROJECT_ROOT, configuration, preprocessor,
                           frames, targets, matrices[configuration], exclusions)
print("Saved core and expanded preprocessing artifacts.")